In [ ]:
import math
import random


# ============================================================
# ACTIVATION FUNCTIONS
# ============================================================

def sigmoid(x):
    # Prevent overflow
    x = max(min(x, 500), -500)
    return 1 / (1 + math.exp(-x))


def sigmoid_derivative(output):
    return output * (1 - output)


# ============================================================
# DNN WITH 2 HIDDEN LAYERS
# ============================================================

class DNN:

    def __init__(self, input_size, hidden1_size,
                 hidden2_size, output_size,
                 learning_rate=0.5):

        self.learning_rate = learning_rate

        # ----------------------------------------------------
        # Layer 1: Input -> Hidden Layer 1
        # ----------------------------------------------------

        self.W1 = [
            [random.uniform(-1, 1) for _ in range(input_size)]
            for _ in range(hidden1_size)
        ]

        self.b1 = [0.0 for _ in range(hidden1_size)]

        # ----------------------------------------------------
        # Layer 2: Hidden Layer 1 -> Hidden Layer 2
        # ----------------------------------------------------

        self.W2 = [
            [random.uniform(-1, 1) for _ in range(hidden1_size)]
            for _ in range(hidden2_size)
        ]

        self.b2 = [0.0 for _ in range(hidden2_size)]

        # ----------------------------------------------------
        # Layer 3: Hidden Layer 2 -> Output
        # ----------------------------------------------------

        self.W3 = [
            [random.uniform(-1, 1) for _ in range(hidden2_size)]
            for _ in range(output_size)
        ]

        self.b3 = [0.0 for _ in range(output_size)]


    # ========================================================
    # FORWARD PROPAGATION
    # ========================================================

    def forward(self, X):

        # ----------------------------------------------------
        # Hidden Layer 1
        # ----------------------------------------------------

        z1 = []

        for i in range(len(self.W1)):

            value = self.b1[i]

            for j in range(len(X)):
                value += self.W1[i][j] * X[j]

            z1.append(value)

        a1 = [sigmoid(x) for x in z1]


        # ----------------------------------------------------
        # Hidden Layer 2
        # ----------------------------------------------------

        z2 = []

        for i in range(len(self.W2)):

            value = self.b2[i]

            for j in range(len(a1)):
                value += self.W2[i][j] * a1[j]

            z2.append(value)

        a2 = [sigmoid(x) for x in z2]


        # ----------------------------------------------------
        # Output Layer
        # ----------------------------------------------------

        z3 = []

        for i in range(len(self.W3)):

            value = self.b3[i]

            for j in range(len(a2)):
                value += self.W3[i][j] * a2[j]

            z3.append(value)

        a3 = [sigmoid(x) for x in z3]


        return z1, a1, z2, a2, z3, a3


    # ========================================================
    # BACKPROPAGATION
    # ========================================================

    def backward(self, X, y, cache):

        z1, a1, z2, a2, z3, a3 = cache

        # ----------------------------------------------------
        # Output Layer Error
        # ----------------------------------------------------

        delta3 = []

        for i in range(len(a3)):

            error = a3[i] - y[i]

            delta = error * sigmoid_derivative(a3[i])

            delta3.append(delta)


        # ----------------------------------------------------
        # Hidden Layer 2 Error
        # ----------------------------------------------------

        delta2 = []

        for j in range(len(a2)):

            error = 0

            for i in range(len(delta3)):
                error += self.W3[i][j] * delta3[i]

            delta = error * sigmoid_derivative(a2[j])

            delta2.append(delta)


        # ----------------------------------------------------
        # Hidden Layer 1 Error
        # ----------------------------------------------------

        delta1 = []

        for j in range(len(a1)):

            error = 0

            for i in range(len(delta2)):
                error += self.W2[i][j] * delta2[i]

            delta = error * sigmoid_derivative(a1[j])

            delta1.append(delta)


        # ====================================================
        # UPDATE OUTPUT LAYER WEIGHTS AND BIAS
        # ====================================================

        for i in range(len(self.W3)):

            for j in range(len(self.W3[i])):

                self.W3[i][j] -= (
                    self.learning_rate
                    * delta3[i]
                    * a2[j]
                )

            self.b3[i] -= (
                self.learning_rate * delta3[i]
            )


        # ====================================================
        # UPDATE HIDDEN LAYER 2
        # ====================================================

        for i in range(len(self.W2)):

            for j in range(len(self.W2[i])):

                self.W2[i][j] -= (
                    self.learning_rate
                    * delta2[i]
                    * a1[j]
                )

            self.b2[i] -= (
                self.learning_rate * delta2[i]
            )


        # ====================================================
        # UPDATE HIDDEN LAYER 1
        # ====================================================

        for i in range(len(self.W1)):

            for j in range(len(self.W1[i])):

                self.W1[i][j] -= (
                    self.learning_rate
                    * delta1[i]
                    * X[j]
                )

            self.b1[i] -= (
                self.learning_rate * delta1[i]
            )


    # ========================================================
    # TRAINING
    # ========================================================

    def train(self, X, Y, epochs):

        for epoch in range(1, epochs + 1):

            total_loss = 0

            for x, y in zip(X, Y):

                # Forward propagation
                cache = self.forward(x)

                prediction = cache[-1]

                # Binary cross entropy
                p = max(min(prediction[0], 1 - 1e-15), 1e-15)

                loss = -(
                    y[0] * math.log(p)
                    + (1 - y[0]) * math.log(1 - p)
                )

                total_loss += loss

                # Backpropagation
                self.backward(x, y, cache)


            # Print only selected epochs
            if epoch == 1 or epoch % 1000 == 0:

                average_loss = total_loss / len(X)

                print(
                    f"Epoch {epoch:5d} | "
                    f"Loss = {average_loss:.6f}"
                )


    # ========================================================
    # PREDICTION
    # ========================================================

    def predict(self, X):

        cache = self.forward(X)

        prediction = cache[-1][0]

        predicted_class = 1 if prediction >= 0.5 else 0

        return prediction, predicted_class


# ============================================================
# XOR DATASET
# ============================================================

X = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

Y = [
    [0],
    [1],
    [1],
    [0]
]


# ============================================================
# CREATE DNN
# ============================================================

random.seed(42)

model = DNN(
    input_size=2,
    hidden1_size=4,
    hidden2_size=3,
    output_size=1,
    learning_rate=0.5
)


# ============================================================
# INITIAL PARAMETERS
# ============================================================

print("=" * 70)
print("             DEEP NEURAL NETWORK - BACKPROPAGATION")
print("=" * 70)

print("\nNetwork Architecture:")
print("Input Layer  : 2 neurons")
print("Hidden Layer 1: 4 neurons")
print("Hidden Layer 2: 3 neurons")
print("Output Layer : 1 neuron")

print("\nTraining Parameters:")
print("Learning Rate:", model.learning_rate)
print("Epochs        : 10000")


# ============================================================
# TRAIN
# ============================================================

print("\nTraining Progress:")
print("-" * 70)

model.train(
    X,
    Y,
    epochs=10000
)


# ============================================================
# FINAL PREDICTIONS
# ============================================================

print("\n")
print("=" * 70)
print("                    FINAL PREDICTIONS")
print("=" * 70)

print(
    f"{'X1':^8} | {'X2':^8} | "
    f"{'Target':^10} | {'Prediction':^12} | {'Class':^8}"
)

print("-" * 70)

for x, target in zip(X, Y):

    prediction, predicted_class = model.predict(x)

    print(
        f"{x[0]:^8} | {x[1]:^8} | "
        f"{target[0]:^10} | "
        f"{prediction:^12.6f} | "
        f"{predicted_class:^8}"
    )

print("-" * 70)


# ============================================================
# FINAL WEIGHTS AND BIASES
# ============================================================

print("\nFINAL LEARNED PARAMETERS")
print("=" * 70)

print("\nHidden Layer 1 Weights:")
for row in model.W1:
    print([round(x, 4) for x in row])

print("\nHidden Layer 1 Bias:")
print([round(x, 4) for x in model.b1])


print("\nHidden Layer 2 Weights:")
for row in model.W2:
    print([round(x, 4) for x in row])

print("\nHidden Layer 2 Bias:")
print([round(x, 4) for x in model.b2])


print("\nOutput Layer Weights:")
for row in model.W3:
    print([round(x, 4) for x in row])

print("\nOutput Layer Bias:")
print([round(x, 4) for x in model.b3])

print("=" * 70)

             DEEP NEURAL NETWORK - BACKPROPAGATION

Network Architecture:
Input Layer  : 2 neurons
Hidden Layer 1: 4 neurons
Hidden Layer 2: 3 neurons
Output Layer : 1 neuron

Training Parameters:
Learning Rate: 0.5
Epochs        : 10000

Training Progress:
----------------------------------------------------------------------
Epoch     1 | Loss = 0.722455
Epoch  1000 | Loss = 0.710410
Epoch  2000 | Loss = 0.590435
Epoch  3000 | Loss = 0.076086
Epoch  4000 | Loss = 0.034298
Epoch  5000 | Loss = 0.023989
Epoch  6000 | Loss = 0.019106
Epoch  7000 | Loss = 0.016193
Epoch  8000 | Loss = 0.014230
Epoch  9000 | Loss = 0.012802
Epoch 10000 | Loss = 0.011709


                    FINAL PREDICTIONS
   X1    |    X2    |   Target   |  Prediction  |  Class  
----------------------------------------------------------------------
   0     |    0     |     0      |   0.010173   |    0    
   0     |    1     |     1      |   0.990394   |    1    
   1     |    0     |     1      |   0.989443   |    